# Working with LangChain

Before LangChain, most developers wrote messy, hardcoded prompts and manual string formatting. LangChain was created to solve the orchestration problem, the pain of coordinating prompts, models, memory, and logic when building real LLM applications.

## **LangChain**


LangChain is a framework that simplifies the development of LLM applications LangChain offers a suite of tools, components, and interfaces that simplify the construction of LLM-centric applications. LangChain enables developers to build applications that can generate creative and contextually relevant content LangChain provides an LLM class designed for interfacing with various language model providers, such as OpenAI, Gemini, and Hugging Face.


LangChain's versatility and flexibility enable seamless integration with various data sources, making it a comprehensive solution for creating advanced language model-powered applications.

<br>

LangChain's open-source framework is available to build applications in Python or JavaScript/TypeScript. Its core design principle is composition and modularity. 

By combining modules and components, one can quickly build complex LLM-based applications. LangChain is an open-source framework that makes it easier to build powerful and personalisable applications with LLMs relevant to user's interests and needs. It connects to external systems to access information required to solve complex problems. 

It also provides abstractions for most of the functionalities needed for building an LLM application and also has integrations that can readily read and write data, reducing the development speed of the application. LangChains's framework allows for building applications that are agnostic to the underlying language model. With its ever expanding support for various LLMs, LangChain offers a unique value proposition to build applications and iterate continuosly.

<br>

The LangChain framework revolves around the following building blocks:
* Model I/O: Interface with language models (LLMs & Chat Models, Prompts, Output Parsers)
* Retrieval: Interface with application-specific data (Document loaders, Document transformers, Text embedding models, Vector stores, Retrievers)
* Chains: Construct sequences/chains of LLM calls
* Memory: Persist application state between runs of a chain
* Agents: Let chains choose which tools to use given high-level



References:
* [LangChain Models Python Documentation](https://docs.langchain.com/oss/python/langchain/models)
* [LangChain Models Python API Reference](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model)

##### Install and Import necessary libraries

In [9]:
# %pip install -q --upgrade \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-google-genai \
#     langchain-huggingface\
#     pandas \
#     python-dotenv

In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from huggingface_hub import InferenceClient

load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = "gemini-2.5-flash-lite"  

if GOOGLE_API_KEY:
    print(f"Raw Gemini client initialized with model: {MODEL_ID}")


HF_TOKEN = os.getenv("HF_TOKEN")
hf_client = InferenceClient(api_key=HF_TOKEN)
HF_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"  

if HF_TOKEN:
    print(f"HF client initialized with model: {HF_MODEL_ID}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Raw Gemini client initialized with model: gemini-2.5-flash-lite
HF client initialized with model: meta-llama/Llama-3.2-3B-Instruct


##### Load the Dataset

In [2]:
import pandas as pd
df = pd.read_csv('products.csv')
df

,product_id,name,category,price,description,rating,stock
0,101,Classic White Sneakers,Footwear,2499,Premium white sneakers with breathable mesh.,4.6,45
1,102,Slim Fit Denim Jeans,Bottoms,1899,High-quality slim fit jeans with perfect stretch.,4.4,32
2,103,Oversized Cotton Hoodie,Tops,1499,Ultra-soft oversized hoodie for casual style.,4.7,28
3,104,Leather Crossbody Bag,Accessories,3299,Elegant genuine leather crossbody bag.,4.8,15
4,105,High Waist Pleated Skirt,Bottoms,2199,Trendy high-waist pleated midi skirt.,4.3,22
5,106,Cashmere Wool Sweater,Tops,4499,Luxurious cashmere blend sweater.,4.9,18
6,107,Vintage Leather Jacket,Outerwear,8999,Timeless black leather jacket.,4.5,12
7,108,Athletic Running Shorts,Bottoms,999,Lightweight athletic shorts.,4.2,50
8,109,Silk Blend Blouse,Tops,2799,Elegant silk-blend blouse.,4.6,35
9,110,Wide Leg Tailored Trousers,Bottoms,3299,Sophisticated wide-leg trousers.,4.7,27


### Raw API Call

Let's first see how we would do this **without LangChain**.

In [ ]:
sys_prompt = f"""You are a helpful fashion stylist.
Here are available products:
{df.to_string(index=False)}
"""
user_prompt = "I am a college student looking for comfortable casual wear under 3000 rupees. Recommend 2 best products with short reason."

In [ ]:
response = hf_client.chat.completions.create(
    model=HF_MODEL_ID,
    messages=[
            {
                "role": "system",
                "content": sys_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
    ],
)

In [16]:
print(response.choices[0].message.content)

Based on your requirement of comfortable casual wear under 3000 rupees, I recommend the following two products:

1. **Slim Fit Denim Jeans (Product ID: 102)**: A high-quality, comfortable, and stylish pair of jeans that fits perfectly. It's perfect for casual wear and can be dressed up or down.
2. **Oversized Cotton Hoodie (Product ID: 103)**: A cozy, ultra-soft oversized hoodie that's ideal for college life. It's perfect for lounging around campus, running errands, or just relaxing with friends.

Both products are comfortable, stylish, and affordable, with prices ranging from 1899 to 1499 rupees, respectively.


### **Using LangChain to Call LLMs**

#### Model Wrappers

LangChain provides an easy out-of-the box support to work with LLMs. It abstracts the actual LLM calls through its own framework. LangChain provides interfaces and integrations for two classes of LLM models
*   **LLMs**: Models that take a text string as input and return a text string
*   **Chat models**: Models that are backed by a language model but take a list of Chat Messages as input and return a Chat Message.

LLMs and chat models are subtly but importantly different. LLMs in LangChain refer to pure "text completion models" (`BaseLLM`) - where a string prompt is taken as the input and the LLM outputs a string. These type of models are now majorly considered outdated and legacy. Most providers have moved towards exposing everything as chat-style APIs, and internally wrap chat APIs for the base LLMs anyway.

Chat Models are LLMs that have been tuned specifically for having turn-based conversations. Instead of a single string, they work well with a list of chat messages as input. Usually these models have labelled messages such as "System", "User" and provides a AI chat message ("Assistant") as the output.

---

Now, chat models are wrapped in LangChain mainly in two ways
* **Direct wrappers from providers**, for example `ChatGoogleGenerativeAI` or `ChatOpenAI`, which are model specific and coupled better with the provider-specific features and parameters. These also get updates with the base models themselves, and are more suitable for production pipelines committed to a particular model.
* **`init_chat_model`**: a provider-agnostic and generic model wrapper, which can be configured to use the provided model and provider names. This gives more independence to switch between providers, but gets updates later as compared to direct wrappers. This however aligns more closely with LangChain's architectural direction.

Let's see an example of how we can use the `ChatGoogleGenerativeAI` wrapper

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [19]:
prompt= f"""You are an expert fashion stylist.

Available products:
{df.to_string(index=False)}

User Query: "I am a college student looking for comfortable casual wear under 3000 rupees. Recommend the 2 most suitable products with short reasoning."""

In [23]:
# Create a response object using the ChatGoogleGenerativeAI function

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    max_tokens=500,
    google_api_key=GOOGLE_API_KEY  
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


##### **Invoking a Model**

In LangChain, models are treated as callable components that expose a standard execution interface. Rather than interacting with provider-specific SDK methods, LangChain models are executed using the `invoke()` method.

The `invoke()` method represents a single, synchronous model call. It accepts one logical input, such as a prompt string or a structured set of chat messages, and returns the model’s corresponding output in its native format (for example, a text string or an AI message).

In [25]:
response = llm.invoke(prompt)

print(response.content)

Here are two comfortable and stylish casual wear options under 3000 rupees, perfect for a college student:

1.  **Oversized Cotton Hoodie (Product ID: 103)**
    *   **Reasoning:** This hoodie is ultra-soft and oversized, making it incredibly comfortable for everyday wear around campus. Its casual style is perfect for relaxed days, and at 1499 rupees, it's well within your budget.

2.  **Slim Fit Denim Jeans (Product ID: 102)**
    *   **Reasoning:** High-quality denim with stretch offers both comfort and a flattering fit. These jeans are a versatile staple that can be easily dressed up or down, making them ideal for a college student's wardrobe. Priced at 1899 rupees, they are a great value for their quality and durability.


Similarly, we can do this for other available model wrappers as well. For example, `ChatOpenAI()` or `ChatHuggingFace()`

In [5]:
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface.llms import HuggingFaceEndpoint  # we also need to use 'HuggingFaceEndpoint' to define HF models

llm = ChatHuggingFace(
    llm=HuggingFaceEndpoint(
        repo_id=HF_MODEL_ID,
        temperature=0.7,
        max_new_tokens=500,
        huggingfacehub_api_token=HF_TOKEN
    )
)

prompt = [{
                "role": "system",
                "content": sys_prompt
            },{
                "role": "user",
                "content": user_prompt
            }]

response = llm.invoke(prompt)

print(response.content)

As a college student, I'd be happy to recommend two comfortable casual wear products that fit your budget of 3000 rupees. Here are my suggestions:

1. **Classic White Sneakers (Product ID: 101)** - Perfect for casual college days, these sneakers are comfortable, breathable, and stylish. Their price is just 2499 rupees, making them an excellent value for the quality and comfort they offer.

2. **Wide Leg Tailored Trousers (Product ID: 110)** - These trousers are both stylish and comfortable, making them ideal for college life. They're also a great investment piece that can be worn with various tops and shoes. Their price is 3299 rupees, but considering the quality and versatility, they're definitely worth the investment.

Both of these products will provide you with comfortable and stylish options for your daily college life, and they're within your budget of 3000 rupees!


##### **`init_chat_model`**

Initialise a chat model from any supported provider using a unified interface. Two main use cases:

1. Fixed model – specify the model upfront and get a ready-to-use chat model
2. Configurable model – choose to specify parameters (including model name) at runtime via `config`. Makes it easy to switch between models/providers without changing your code

The syntax is as follows

```Python
init_chat_model(
  model: str | None = None,
  *,
  model_provider: str | None = None,
  configurable_fields: Literal['any'] | list[str] | tuple[str, ...] | None = None,
  config_prefix: str | None = None,
  **kwargs: Any = {}
) -> BaseChatModel | _ConfigurableModel
```

Usually when setting up a client, we can provide the API key explicitly. However, the client also looks for the keys in the environment before requiring them as explicit arguments. If the expected variable is present, you do not need to pass the key in code.

OpenAI → `OPENAI_API_KEY`

Google Gemini → `GOOGLE_API_KEY`

Hugging Face → `HF_TOKEN`

`init_chat_model` also looks for API keys automatically for whatever model you request.

In [ ]:
from langchain.chat_models import init_chat_model

gemini = init_chat_model(model="gemini-2.5-flash-lite", model_provider="google_genai")

gemini.invoke("what's your name").content

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


'I do not have a name. I am a large language model, trained by Google.'

In [ ]:
# Another way of doing this is providing the model as "(provider:model)"

# gemini = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0)

---

As mentioned above, some of the parameters can also be set as configurable

In [ ]:
# !pip install langchain langchain-openai

In [ ]:
# We don't need to specify configurable=True if a model isn't specified
configurable_model = init_chat_model(temperature=0)


# Use GPT-4o to generate the response (will work if a key is loaded in the environment; any model can be used)
configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "gpt-4o-mini"}},      # configurable fields are quite useful
)

AIMessage(content='I’m called ChatGPT. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 11, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DDrsxkRbirSUAwh1m2ZuqgbngtX9X', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c9f37-f2ab-7d30-80b8-cd242d4f3703-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 13, 'total_tokens': 24, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [55]:
configurable_model_with_default = init_chat_model(
    "openai:gpt-4o",
    configurable_fields="any",  # This allows us to configure other params like temperature, max_tokens, etc at runtime.
    config_prefix="foo",        # A configured field will be identified with the `config_prefix`
)

# GPT-4o response with temperature 0 (as set in default)
configurable_model_with_default.invoke("what's your name").content

"I'm an AI language model created by OpenAI, and I don't have a personal name. You can call me Assistant! How can I help you today?"

In [56]:
configurable_model_with_default.invoke(
    "what's your name",
    config={
        "configurable": {
            "foo_max_tokens": 5,
        }
    },
).content

"I'm called ChatGPT."

#### Runnables

In LangChain, **Runnables** are the core abstraction for any executable component in an LLM pipeline. A runnable is simply an object that takes an input and produces an output, exposing a common execution interface such as `invoke()`. Models, prompt templates, and chains are all implemented as runnables, which allows them to be composed, reused, and combined in a uniform way.

Because everything follows the same runnable contract, LangChain can treat simple model calls and complex multi-step workflows in a consistent manner.

In [ ]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0)

# The model itself is a Runnable
# response = llm.invoke("What is gradient descent?")
# print(response.content)

Here, the chat model acts as a runnable: it accepts an input, executes once via `invoke()`, and returns an output. More complex systems are built by composing multiple runnables together, but the execution principle remains the same.

**Raw SDK** → You write provider-specific code (`genai.Client`, `generate_content`, `GenerationConfig`)

**LangChain** → One consistent interface (`ChatGoogleGenerativeAI`, `.invoke()`, `temperature` as param)

> LangChain provides a standardised interface.

Example: To switch to OpenAI later → just `from langchain_openai import ChatOpenAI` and `llm = ChatOpenAI(model="gpt-4o-mini")`, or simply change the model configuration. No rewriting of prompts, chains, memory, or parsers.

## Model I/O


LangChain's Model I/O component provides support to interface with the LLM and generate responses.
The Model I/O consists of:
* **Prompts**: Templatise, dynamically select, and manage model inputs
* **Document Loaders**: To ingest external files in a compatible manner
* **Output Parsers**: Extract information from model outputs

### **Prompts and Prompt Templates**

A prompt is the input instruction given to a language model. In simple cases, this can be a plain string passed directly at invocation time. However, for reusable and structured interactions, LangChain provides **prompt templates**, which allow variables, roles, and formatting to be defined explicitly.

Prompt templates make prompts parameterised, repeatable, and composable, which is essential when building chains or agent workflows.

We have seen the simplest type of prompts, which can be passed directly while invoking.

In [13]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", temperature=0, max_tokens=100)

response = llm.invoke("Explain regularisation in machine learning.")
print(response.content)

Regularization is a technique used in machine learning to prevent overfitting, which occurs when a model learns the noise in the training data rather than the underlying patterns. Overfitting can lead to poor generalization to new, unseen data. Regularization introduces additional information or constraints into the model to encourage it to be simpler and more robust.

There are several common types of regularization techniques:

1. **L1 Regularization (Lasso)**:
   - L1 regularization adds a penalty equal


This is suitable for quick experiments but does not scale well. 

#### `PromptTemplate`

To build a more reusable and consistent prompt which works well with dynamic content, let's try using a `PromptTemplate`. Refer to the [documentation here](https://reference.langchain.com/python/langchain-core/prompts/prompt/PromptTemplate).

In [18]:
from langchain_core.prompts import PromptTemplate

llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0)

# Use the `PromptTemplate.from_template()` method to define a template

prompt = PromptTemplate.from_template("Here are available products:{data}. Suggest me something casual under 3000",
                                      template_format="f-string")


# Now just provide the value to the data variable using format()

formatted_prompt = prompt.format(data=df.to_string(index=False))

response = llm.invoke(formatted_prompt)
print(response.content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Here are some casual product suggestions under 3000:

*   **Classic White Sneakers** (Product ID: 101) - Price: 2499
*   **Slim Fit Denim Jeans** (Product ID: 102) - Price: 1899
*   **Oversized Cotton Hoodie** (Product ID: 103) - Price: 1499
*   **High Waist Pleated Skirt** (Product ID: 105) - Price: 2199
*   **Athletic Running Shorts** (Product ID: 108) - Price: 999


#### `ChatPromptTemplate`

More recently, with the move towards chat models, the `ChatPromptTemplate` ([documentation](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate)) has instead become to recommended template style. It aligns better with the chat-based models and provides a multi-turn conversation based input. We use the `.from_messages()` method instead of the `.from_template()` here.


We use the roles `"system"`, `"ai"`, and `"human"`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini", temperature=0)

user_name = input("Please enter your name: ")
query = input("Please enter the style or product you wish to buy: ")


prompt = ChatPromptTemplate.from_messages([
                            ("system", "You are a shopping assistant. Here are available products:{data}.\
                                        Start with greeting the user and ask what they want"),
                            ("ai", "Hi {user}, what do you plan to buy?"),
                            ("human", "I want to buy {desired_style}")])


messages = prompt.format_messages(data=df.to_string(index=False), user=user_name, desired_style=query)

response = llm.invoke(messages)
print(response.content)

Great! For casual wear, I recommend the following options:

1. **Oversized Cotton Hoodie** - Price: ₹1499, Rating: 4.7 (Ultra-soft oversized hoodie for casual style)
2. **Classic White Sneakers** - Price: ₹2499, Rating: 4.6 (Premium white sneakers with breathable mesh)
3. **Athletic Running Shorts** - Price: ₹999, Rating: 4.2 (Lightweight athletic shorts)

Would you like more information on any of these items or assistance with something else?


Note that here we are only simulating a conversation to give it to the LLM as context. The user is not actually seeing the LLM's greeting.


If your prompt has only a single input variable (i.e., one instance of `{variable_name}`), and you invoke the template with a non-dict object, the prompt template will inject the provided argument into that variable location.

In [34]:
template = ChatPromptTemplate(
    [
        ("system", "You are a shopping assistant. Here are available products:{data}"), # this can also be done with the "human" message
        ("human", "suggest me something casual"),
    ]
)

prompt_value = template.invoke(df.to_string(index=False))

Note that we invoked the template here, not the LLM. In LangChain, the `.invoke()` method is the standard way to run nearly any Runnable component. So we can invoke models, prompt templates, chains and so on.

Let's see how the prompt is built when invoked.

In [ ]:
# using `prompt_value.content` directly doesn't work here like it does for LLM responses

prompt_value.messages[0].content

'You are a shopping assistant. Here are available products: product_id                       name    category  price                                       description  rating  stock\n        101     Classic White Sneakers    Footwear   2499      Premium white sneakers with breathable mesh.     4.6     45\n        102       Slim Fit Denim Jeans     Bottoms   1899 High-quality slim fit jeans with perfect stretch.     4.4     32\n        103    Oversized Cotton Hoodie        Tops   1499     Ultra-soft oversized hoodie for casual style.     4.7     28\n        104      Leather Crossbody Bag Accessories   3299            Elegant genuine leather crossbody bag.     4.8     15\n        105   High Waist Pleated Skirt     Bottoms   2199             Trendy high-waist pleated midi skirt.     4.3     22\n        106      Cashmere Wool Sweater        Tops   4499                 Luxurious cashmere blend sweater.     4.9     18\n        107     Vintage Leather Jacket   Outerwear   8999                

You can see that for prompt templates with just a single variable, you can directly invoke the template. For most chat models, you can pass the prompt value directly while invoking the model (`llm.invoke(prompt_value)`). For others, you can simply extract the content and invoke (`llm.invoke(prompt_value.messages[0].content)`).

#### Messages

For reusable, dynamic prompts, we can use the prompt templates. But for a more chat-style exchange, we can use messages.

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

Messages contain
* Role - Identifies the message type (e.g. system, user)
* Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
* Metadata - Optional fields such as response information, message IDs, and token usage


We have three types of roles:
1. `SystemMessage` – sets global behaviour or instructions
2. `HumanMessage` – represents user input
3. `AIMessage` – represents model output

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a machine learning tutor."),
    HumanMessage("Explain regularisation.")     # either use the keyword or directly pass as positional arg
]

# You can also define the messages externally
system_msg = SystemMessage("You are a machine learning tutor.")

messages = [system_msg, HumanMessage("Explain regularisation.")]

llm.invoke(messages).content

"Regularization is a technique used in machine learning and statistics to prevent overfitting, which occurs when a model learns the noise in the training data rather than the underlying patterns. Overfitting can lead to poor generalization to new, unseen data. Regularization introduces additional information or constraints into the model to help it generalize better.\n\nThere are several common types of regularization techniques:\n\n1. **L1 Regularization (Lasso)**:\n   - L1 regularization adds a penalty equal to the absolute value of the magnitude of coefficients to the loss function. This can lead to sparse models where some coefficients are exactly zero, effectively performing feature selection.\n   - The regularization term is given by: \n     \\[\n     \\text{Penalty} = \\lambda \\sum_{i=1}^{n} |w_i|\n     \\]\n   - Here, \\( \\lambda \\) is the regularization parameter that controls the strength of the penalty, and \\( w_i \\) are the model coefficients.\n\n2. **L2 Regularization

Apart from the message content, we can also pass some metadata with the messages. For example

In [ ]:
system_msg = SystemMessage("You are a helpful assistant.")

human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users; how it works depends on the llm provider
    id="msg_123",  # Optional: unique identifier for tracing
)

messages = [system_msg, human_msg]

llm.invoke(messages).content

'Hello, Alice! How can I assist you today?'

An `AIMessage` represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access. It is sometimes helpful to manually create a new `AIMessage` object manually and insert it into the message history as if it came from the model.

In [42]:
from langchain_core.messages import AIMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

You can also specify the same messages as role-content dictionaries. 

Note that the role names passed in the dict style messages are not hard-bound. You can use any role name, as long as the roles are clearly understandable to the model.

In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]

llm.invoke(messages).content

'Cherry blossoms bloom,  \nWhispers of a gentle breeze,  \nLife awakens bright.'

Earlier, we simulated the conversation memory of the shopping assistant (`ChatPromptTemplate`) to pass it to the model. Using messages, now let's try having an actual conversation between user and LLM to greet the user and suggest some products.

In [ ]:
messages = [
    SystemMessage(content=f"You are a shopping assistant. Here are available products:\n{df.to_string(index=False)}.\
                            Greet the user and ask what they are looking to buy")
    ]

print("Assistant: Hello! What is your name?")

while True:
    user_input = input("User: ")
    print("User:", user_input)

    if user_input.lower() in {"exit", "quit"}: 
        break

    messages.append(HumanMessage(content=user_input))

    response = llm.invoke(messages)

    print("Assistant:", response.content)
    messages.append(AIMessage(content=response.content))

Assistant: Hello! What is your name?
User: john
Assistant: Hello John! How can I assist you today? What are you looking to buy?
User: something casual and cheap
Assistant: For something casual and affordable, I recommend the following options:

1. **Athletic Running Shorts** - Price: 999
   - Lightweight athletic shorts, perfect for casual wear or workouts. Rating: 4.2 (Stock: 50)

2. **Oversized Cotton Hoodie** - Price: 1499
   - Ultra-soft oversized hoodie for a relaxed style. Rating: 4.7 (Stock: 28)

Would you like more information on any of these items, or do you have something else in mind?
User: exit


### **Document Loaders**

In real applications, product data usually lives in files such as CSV, JSON, or PDFs. Manually copying data into prompts is error-prone and not scalable.

**Document loaders** are components in LangChain that ingest raw data from external sources and convert it into a standard `Document` format.

Common types of document loaders


1. **File-based loaders**

    Used for local or uploaded files

    * `TextLoader` – plain `.txt` files
    * `PyPDFLoader`, `PDFPlumberLoader` – PDF documents
    * `Docx2txtLoader` – Word (.docx) files
    * `CSVLoader` – CSV files
    * `UnstructuredFileLoader` – generic loader for mixed formats

2. **Web-based loaders**

    Used to pull content from URLs

    * `WebBaseLoader` – static web pages
    * `SitemapLoader` – multiple pages via sitemap
    * `PlaywrightURLLoader` – dynamic, JavaScript-heavy pages

3. **Code and notebook loaders**

    Used for technical repositories

    * `PythonLoader`, `DirectoryLoader` – source code files
    * `NotebookLoader` – Jupyter notebooks

4. **Cloud and platform loaders**

    Used for third-party data sources like `GoogleDriveLoader`

5. **Database loaders**

    Used to load structured data as text

    * `SQLDatabaseLoader`
    * `DataFrameLoader`


<br>

> Document loaders focus only on **data ingestion**, not chunking or retrieval. Splitting, embedding, and indexing are intentionally handled by separate components to keep the pipeline modular.

Document loaders become a natural step in retrieval based pipelines, to ingest documents and then embed them as vectors. We will discuss that later. For now, let's just see an example on how we can use a CSV loader to simply use it in our shopping assistant.

**Document**

LangChain defines a schema specially for working with Documents. A Document is a piece of text and associated metadata.

In [43]:
from langchain_core.documents import Document

document = Document(
    page_content="Hello, world!", metadata={"source": "https://example.com"}
)

document

Document(metadata={'source': 'https://example.com'}, page_content='Hello, world!')

Documents and loaders are being demonstrated here only for showing ingestion. As per LangChain, **Document is for retrieval workflows** (e.g., RAG), not chat I/O. For sending text to an LLM in a conversation, simply use message types.

In [44]:
from langchain_community.document_loaders import CSVLoader

In [ ]:
loader = CSVLoader(file_path="products.csv")
documents = loader.load()

print(f"Total documents loaded: {len(documents)}")

# Each row becomes one document
print("\nFirst document preview:")
print(documents[0].page_content[:500])

Total documents loaded: 10

First document preview:
product_id: 101
name: Classic White Sneakers
category: Footwear
price: 2499
description: Premium white sneakers with breathable mesh.
rating: 4.6
stock: 45


**Injecting Loaded Documents into Prompts**

We can now combine the loaded documents with our prompt template. This approach keeps the code clean and makes it easy to update the data source without changing the prompt logic.

In [49]:
prompt_template = ChatPromptTemplate.from_template(
    """You are an expert fashion stylist.

Here is the complete list of available products:
{products}

User Query: {query}

Recommend the 2 most suitable products with short reasoning."""
)

products_data = "\n\n".join([doc.page_content for doc in documents])

user_query = "I am a college student looking for comfortable casual wear under 3000 rupees."

prompt = prompt_template.format(products=products_data, query=user_query)

print(llm.invoke(prompt).content)

Based on your requirements for comfortable casual wear under 3000 rupees, I recommend the following two products:

1. **Oversized Cotton Hoodie (Product ID: 103)**
   - **Price:** 1499
   - **Description:** Ultra-soft oversized hoodie for casual style.
   - **Reasoning:** This hoodie is perfect for a college student looking for comfort and style. Its oversized fit makes it easy to wear, and the soft cotton material is ideal for everyday casual wear.

2. **Slim Fit Denim Jeans (Product ID: 102)**
   - **Price:** 1899
   - **Description:** High-quality slim fit jeans with perfect stretch.
   - **Reasoning:** These jeans offer a stylish yet comfortable fit, making them versatile for various casual occasions. The stretch fabric ensures ease of movement, which is great for a busy college lifestyle.

Both options are stylish, comfortable, and well within your budget!


- Document loaders allow us to read external files in a standardised way, but these are originally intended to be used with retrieval pipelines
- We can directly pass the loaded content into prompts without manual string formatting
- This pattern becomes very important when working with large datasets or multiple file types

We have now successfully loaded real product data and used it in our assistant. Now, let's learn how to control the output format using Output Parsers so the assistant returns structured data instead of free text.

### **Output Parsers**

So far our assistant returns free-form text. In real applications, we often need the output in a specific structured format such as JSON, Python dictionary, or a clean list of objects. This allows us to reliably parse the response and use it in downstream code, databases, or front-end applications.

LangChain provides [Output Parsers](https://reference.langchain.com/python/langchain-core/output_parsers) to enforce structure on the LLM output.

In this section we will explore some common parsers using our fashion product assistant.

In [ ]:
from langchain_core.output_parsers import (
    StrOutputParser,                # simple string outputs
    JsonOutputParser,               # handling JSON-style outputs
    CommaSeparatedListOutputParser, # handling comma-separated lists
    PydanticOutputParser            # Pydantic outputs
    )

Output Parsers are classes that help structure language model responses. Typically, LLMs output text as responses; however, if you want to get a more structured response than just the response text Output Parsers are effective.

1. **Format Instructions** - A autogenerated prompt that tells the LLM how to format it's response based off your desired result
2. **Parser** - A method which will extract your model's text output into a desired structure

In [55]:
# View the instructions for the CommaSeparateedListOutputParser
list_parser = CommaSeparatedListOutputParser()
# Creating an instance of the CommaSeparatedListOutputParser class to handle comma-separated lists in LLM's output.
list_parser_instructions = list_parser.get_format_instructions()
# The 'list_parser_instructions' variable now holds the format instructions for the parser.
list_parser_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

#### `StrOutputParser` - Basic String Output

This is the simplest parser. It ensures the output is returned as plain text. Often used explicitly to signal “no structure required”.

In [75]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite", max_tokens=200)

prompt = ChatPromptTemplate.from_messages([
                            ("system", "You are a shopping assistant. Here are available products:{data}"),
                            ("human", "I am a college student looking for comfortable casual wear under 3000 rupees")])


messages = prompt.format_messages(data=df.to_string(index=False))

response = llm.invoke(messages)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Now, we just define a `StrOutputParser` and pass the response for it to extract the relevant output text and format it as a string.

In [63]:
# Basic string parser
str_parser = StrOutputParser()

output = str_parser.invoke(response)

print(output)

Here are some comfortable casual wear options under 3000 rupees that would be great for a college student:

*   **Oversized Cotton Hoodie (product_id: 103)** - Price: 1499. This is perfect for a relaxed, casual look and is super comfortable.
*   **Slim Fit Denim Jeans (product_id: 102)** - Price: 1899. A good pair of jeans is a staple for any college student, and these have a great stretch for comfort.
*   **Athletic Running Shorts (product_id: 108)** - Price: 999. If you prefer shorts, these are lightweight and comfortable for everyday wear.
*   **Classic White Sneakers (product_id: 101)** - Price: 2499. These are a versatile and comfortable footwear choice that goes with almost anything.


#### `JsonOutputParser` - Structured JSON Output

We can instruct the model to return JSON and use `JsonOutputParser` to automatically parse it into a Python dictionary.

In [ ]:
JsonOutputParser().get_format_instructions()

'Return a JSON object.'

We need to give additional information here, according to the output we want.

In [ ]:
# Basic string parser
json_parser = JsonOutputParser()

format_instructions = """Return the answer strictly in JSON with the following format:
{{
  "recommended_products": [
    {{
      "product_id": number,
      "name": string,
      "price": number,
      "reason": string
    }}
  ]
}}"""

prompt = ChatPromptTemplate.from_messages([
                            ("system", "You are a shopping assistant. Here are available products:{data} \n\n {instructions}"),
                            ("human", "I am a college student looking for comfortable casual wear under 3000 rupees")])


json_messages = prompt.format_messages(data=df.to_string(index=False), instructions=format_instructions)

json_response = llm.invoke(json_messages)

In [ ]:
json_response

AIMessage(content='```json\n{\n  "recommended_products": [\n    {\n      "product_id": 101,\n      "name": "Classic White Sneakers",\n      "price": 2499,\n      "reason": "Perfect for everyday college wear, offering comfort and style. They are versatile and can be paired with various outfits."\n    },\n    {\n      "product_id": 102,\n      "name": "Slim Fit Denim Jeans",\n      "price": 1899,\n      "reason": "A wardrobe staple that is comfortable for daily wear and can be dressed up or down. The stretch ensures comfort throughout the day."\n    },\n    {\n      "product_id": 103,\n      "name": "Oversized Cotton Hoodie",\n      "price": 1499,\n      "reason": "Ideal for a relaxed college vibe', additional_kwargs={}, response_metadata={'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ca394-7fb6-7fc3-87e8-69d40b2e7545-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43

In [ ]:
output = json_parser.invoke(json_response)

output

{'recommended_products': [{'product_id': 101,
   'name': 'Classic White Sneakers',
   'price': 2499,
   'reason': 'Perfect for everyday college wear, offering comfort and style. They are versatile and can be paired with various outfits.'},
  {'product_id': 102,
   'name': 'Slim Fit Denim Jeans',
   'price': 1899,
   'reason': 'A wardrobe staple that is comfortable for daily wear and can be dressed up or down. The stretch ensures comfort throughout the day.'},
  {'product_id': 103,
   'name': 'Oversized Cotton Hoodie',
   'price': 1499,
   'reason': 'Ideal for a relaxed college vibe'}]}

We can also use the `.parse()`, a low-level method that converts raw text into a structured Python object according to the rules of an output parser. Using `.invoke()` on a parser internally calls the `.parse()` method as well.

In [ ]:
#  The parse() method uses the extracted content instead of the whole response

json_parser.parse(json_response.content)

{'recommended_products': [{'product_id': 101,
   'name': 'Classic White Sneakers',
   'price': 2499,
   'reason': 'These sneakers are perfect for everyday college wear, offering comfort and style. They are versatile and can be paired with various outfits.'},
  {'product_id': 103,
   'name': 'Oversized Cotton Hoodie',
   'price': 1499,
   'reason': "This hoodie is incredibly soft and comfortable, ideal for a relaxed college look. It's a great option for staying warm and stylish between classes."},
  {'product_id': 102, 'name': 'Slim Fit Denim Jeans', 'price': 1899}]}

#### `CommaSeparatedListOutputParser`

This parses outputs into a Python list

In [76]:
# Comma separated list parser
csl_parser = CommaSeparatedListOutputParser()

output = csl_parser.invoke(response)

print(output)

['Great! Based on your preferences', 'here are some comfortable casual wear options under 3000 rupees that would be perfect for a college student:', '*   **Oversized Cotton Hoodie (product_id: 103):** This hoodie is super soft and perfect for a relaxed', "casual look. It's priced at 1499 rupees.", '*   **Slim Fit Denim Jeans (product_id: 102):** These jeans offer a great blend of style and comfort with a bit of stretch. They are priced at 1899 rupees.', "*   **Athletic Running Shorts (product_id: 108):** If you're looking for something even more casual and breathable", 'these lightweight shorts are a good option at 999 rupees.', '*   **Classic White Sneakers (product_id: 101):** To complete a casual outfit', 'these sneakers are a versatile choice. They']


As you can see, it simply split the whole output based on sections. We can again use `format_instructions` to make it return only the products as list. As an exercise, try doing this yourself.

#### `PydanticOutputParser` - Strongly Typed Structured Output

[Pydantic](https://docs.pydantic.dev/latest/) is a popular Python library for data validation and settings management, using Python type hints to enforce data types at runtime.

`PydanticOutputParser` is the most powerful and production-ready option. Instead of parsing free-form text or loosely structured JSON, the model is guided to produce data that conforms to a Pydantic schema, which is then validated at runtime.

##### Defining a Pydantic Class

When using `PydanticOutputParser`, the Pydantic class defines the contract between the language model and your application. It specifies exactly what fields are expected, their data types, and optional constraints or descriptions. Without this class, the parser has no authoritative definition of what a “valid” output looks like.

The parser uses the Pydantic class to automatically generate formatting instructions via `get_format_instructions()`. These instructions tell the model which fields must appear, what type each field should have, and the overall JSON structure to follow.

For example, a Pydantic field such as:

`price: int = Field(description="Price in rupees")` 

is translated into prompt-level guidance that the model must output an integer field called price. 


* Pydantic class defines the schema
* Parser instructions tell the model how to format its output
* Parser validation checks that the output actually matches the schema


Let's create classes to inform the model about the schema and final output

In [80]:
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    product_name: str = Field(description="Exact name of the product")
    reason: str = Field(description="Short reason for recommendation")
    price: int = Field(description="Price in rupees")


# Suppose we want only 2 suggestions, an easy way is to just create another class which gives 2 suggestions
class Recommendations(BaseModel):
    recommendations: list[ProductRecommendation] = Field(description="List of 2 recommended products")

In [83]:
py_parser = PydanticOutputParser(pydantic_object=Recommendations)

format_instructions = py_parser.get_format_instructions()

prompt = ChatPromptTemplate.from_messages([
                            ("system", "You are a shopping assistant. Here are available products:{data} \n\n {instructions}"),
                            ("human", "I am a college student looking for comfortable casual wear under 3000 rupees")])

In [85]:
py_parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"$defs": {"ProductRecommendation": {"properties": {"product_name": {"description": "Exact name of the product", "title": "Product Name", "type": "string"}, "reason": {"description": "Short reason for recommendation", "title": "Reason", "type": "string"}, "price": {"description": "Price in rupees", "title": "Price", "type": "integer"}}, "required": ["product_name", "reason", "price"], "title": "ProductRecommendation", "type": "object"}}, "properties": {"recommendations": {"description": "List of 2 recommended products", "items": {"$ref

In [87]:
py_messages = prompt.format_messages(data=df.to_string(index=False), instructions=format_instructions)

py_response = llm.invoke(py_messages)

In [89]:
output = py_parser.invoke(py_response)

print(output)

recommendations=[ProductRecommendation(product_name='Oversized Cotton Hoodie', reason='Super soft and perfect for a relaxed college look, well within your budget.', price=1499), ProductRecommendation(product_name='Slim Fit Denim Jeans', reason='Versatile and comfortable jeans that can be dressed up or down, a great staple.', price=1899)]


Here, we got two `ProductRecommendation()` in the `recommendations` list like output

So,

- Output parsers solve the problem of unreliable free-text responses from LLMs
- StrOutputParser is used for simple string output
- JsonOutputParser converts output into Python dictionaries
- PydanticOutputParser provides strict typing and validation, making it ideal for production systems
- Using parsers allows us to reliably use the LLM output in code, APIs, or databases

<br>

Note that output parsers emerged as an early solution to the challenge of obtaining structured output from LLMs.

Today, most LLMs support structured output natively. In such cases, using output parsers may be unnecessary, and you should leverage the model's built-in capabilities for structured output. 

Output parsers remain valuable when working with models that do not support structured output natively, or when you require additional processing or validation of the model's output beyond its inherent capabilities (for example, most models provide only JSON style structured outputs).


<br>

We now have full control over both input and output in our fashion assistant.

## Chaining

So far we have been making single LLM calls. Using an LLM in isolation is fine for simple applications, but more complex applications require chaining multiple steps - either with each other or with other components. For example, first generate recommendations, then refine them, summarise them, or validate them.

Writing this manually by calling the LLM multiple times and passing outputs yourself results in messy, hard-to-read, and difficult-to-maintain code.

LangChain provides **Chains** that can be used to combine multiple components together to create a single, coherent application. Chaining lets us connect multiple components such as prompts, models, and parsers into a clean, sequential pipeline.

For example, we can create a chain that takes user input, formats it with a prompt template, and then passes the formatted response to an LLM. We can build more complex chains by combining multiple chains together, or by combining chains with other components.

In modern LangChain, chaining is expressed explicitly using the **LangChain Expression Language (LCEL**), which treats prompts, models, and parsers as composable runnables.

In this section we will chain multiple steps for our fashion assistant.

LCEL treats every component (prompt, model, parser, function) as a Runnable. This allows us to chain, branch, and transform data in a clean and consistent way.

### Basic LCEL Chaining

LCEL provides a simple, declarative syntax using the pipe operator `|` that makes complex workflows readable, composable, and production-ready. It automatically passes the output of one component as input to the next.

In [ ]:
llm = init_chat_model("gpt-4o-mini", max_tokens=250)

prompt_template = ChatPromptTemplate.from_messages([("system", """You are an expert fashion stylist.
                                                   Here is the complete list of available products:
                                                   {products}
                                                   
                                                   Recommend the 2 most suitable products with short reasoning"""), 
                                                   ("human", "{query}")
                                                ])

basic_chain = prompt_template | llm | StrOutputParser()

user_query = "I am a college student looking for comfortable casual wear under 3000 rupees."

response = basic_chain.invoke({"products": df.to_string(index=False), "query": user_query})

print(response)

Based on your requirements for comfortable casual wear under 3000 rupees, I recommend the following two products:

1. **Classic White Sneakers (Price: 2499)**
   - Reasoning: These sneakers are not only stylish but also provide the comfort and breathability needed for all-day wear, making them perfect for college life.

2. **Oversized Cotton Hoodie (Price: 1499)**
   - Reasoning: This ultra-soft hoodie is ideal for a casual and relaxed look. Its comfort and versatility make it suitable for layering over various outfits, perfect for college settings.

Both items fit your budget and are great for a comfortable, casual look while attending classes.


These types of chains were originally called `SequentialChain` in LangChain. Though now, they have been replaced by this composition style expression.

As an exercise, try using the custom `JsonOutputParser` we defined for product recommendations with this chain.

### Control-flow Runnables

The other type of chains were called `RouterChain`, which have also been replaced by conditional logic with runnables. Now explicit control-flow is handled using specialised runnables rather than named chain classes. The most commonly used control-flow runnables are `RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`, and `RunnableBranch`.

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

#### `RunnablePassthrough` - Output Unchanged

`RunnablePassthrough` forwards its input unchanged. It is used when the original input must be preserved and combined with additional computed outputs later in the pipeline.

In [13]:
passthrough = RunnablePassthrough()

passthrough.invoke({"hi"})

{'hi'}

RunnablePassthrough is a pure identity runnable:
* It has no internal steps
* It performs no transformation
* It simply returns its input unchanged

It is commonly used inside `RunnableParallel` to keep raw inputs while other branches compute new values. It has no chains because it is the chain’s identity step.

#### `RunnableParallel` - Running Multiple Steps in Parallel

We can run multiple independent operations at the same time and combine their results.

Let's see an example where one chain will take the prompt to suggest products, while the other chain will contain a different query.

In [ ]:
# Parallel chain: Get recommendation + Get price range summary

summary_prompt = ChatPromptTemplate.from_template("""Here are the products:
                                                  {products}
                                                  Give a short summary of price range for casual wear under 3000 rupees.""")

parallel_chain = RunnableParallel({
    "recommendation": prompt_template | llm | StrOutputParser(),
    "price_summary":  summary_prompt| llm | StrOutputParser()
})

result = parallel_chain.invoke({
    "products": df.to_string(index=False),
    "query": user_query
})

result

{'recommendation': 'Here’s a price summary of products priced under 3000:\n\n1. **Classic White Sneakers** - Price: 2499\n2. **Slim Fit Denim Jeans** - Price: 1899\n3. **Oversized Cotton Hoodie** - Price: 1499\n4. **High Waist Pleated Skirt** - Price: 2199\n5. **Athletic Running Shorts** - Price: 999\n\n**Total of Products under 3000:** \n2499 + 1899 + 1499 + 2199 + 999 = **10,095** \n\n**Average Price of Products under 3000:** \nTotal Price (10,095) / Number of Products (5) = **2019** \n\nSo, the total price of products under 3000 is **10,095**, and the average price is **2019**.',
 'price_summary': 'The price range for casual wear under 3000 rupees includes the following products:\n\n1. **Classic White Sneakers** - ₹2499\n2. **Slim Fit Denim Jeans** - ₹1899\n3. **Oversized Cotton Hoodie** - ₹1499\n4. **High Waist Pleated Skirt** - ₹2199\n5. **Athletic Running Shorts** - ₹999\n\nOverall, the casual wear options available in this price range range from ₹999 to ₹2499.'}

#### `RunnableLambda` - Functions Inside Chains

`RunnableLambda` allows us to add custom Python functions inside the chain. It wraps a plain Python function so it can participate in an LCEL pipeline. It is used for lightweight transformations, conditional checks, or preprocessing logic that does not require an LLM.

In [18]:
length_check = RunnableLambda(lambda x: len(x))

length_check.invoke("Hello world")

11

In [ ]:
def format_output(data):
    return f"Recommendation from Assistant:\n{data['recommendation']}\n\nPrice Summary:\n{data['price_summary']}"

full_lcel_chain = (
    parallel_chain | RunnableLambda(format_output)
)

final_output = full_lcel_chain.invoke({
    "products": df.to_string(index=False),
    "query": user_query
})

print(final_output)

Recommendation from Assistant:
Here’s the price summary of products under 3000:

1. **Classic White Sneakers** - 2499
2. **Slim Fit Denim Jeans** - 1899
3. **Oversized Cotton Hoodie** - 1499
4. **High Waist Pleated Skirt** - 2199
5. **Athletic Running Shorts** - 999

**Total Price of all products under 3000:** 2499 + 1899 + 1499 + 2199 + 999 = **10095**

Price Summary:
For casual wear under 3000 rupees, the price range includes:

1. Oversized Cotton Hoodie - ₹1499
2. Slim Fit Denim Jeans - ₹1899
3. High Waist Pleated Skirt - ₹2199
4. Athletic Running Shorts - ₹999

The prices for these items range from ₹999 to ₹2199.


#### `RunnableBranch`

RunnableBranch enables conditional routing. It selects exactly one runnable to execute based on a condition evaluated at runtime. This is the modern replacement for `RouterChain`.

Each branch consists of:
* a condition (callable returning True or False)
* a runnable to execute if the condition matches

In [ ]:
# Let's redefine the summary prompt a bit (RunnableBranch works with similar inputs for all branches)
summary_prompt = ChatPromptTemplate.from_messages([
                                                    ("system", """Here are the products: {products}"""),
                                                    ("human", "{query}")
                                                ])

router = RunnableBranch(
    (
        lambda x: "summary" in x["query"].lower(),
        summary_prompt | llm | StrOutputParser()
    ),
    (
        lambda x: "recommend" in x["query"].lower(),
        prompt_template | llm | StrOutputParser()
    ),
    llm | StrOutputParser()  # default branch
)

user_query = "give a short summary of products under 3000"

# user_query = "recommend something casual under 3000"  # this will run the other branch

output = router.invoke({
    "products": df.to_string(index=False),
    "query": user_query
})

print(output)

Here’s a summary of products under 3000:

1. **Classic White Sneakers** - Price: 2499, Rating: 4.6, Stock: 45. Premium breathable mesh sneakers ideal for casual wear.
   
2. **Slim Fit Denim Jeans** - Price: 1899, Rating: 4.4, Stock: 32. High-quality jeans featuring perfect stretch for comfort.

3. **Oversized Cotton Hoodie** - Price: 1499, Rating: 4.7, Stock: 28. An ultra-soft hoodie perfect for a casual style.

4. **High Waist Pleated Skirt** - Price: 2199, Rating: 4.3, Stock: 22. A trendy midi skirt that offers a stylish silhouette.

5. **Athletic Running Shorts** - Price: 999, Rating: 4.2, Stock: 50. Lightweight shorts designed for athletic and casual use.

Overall, these products combine style, comfort, and versatility at affordable prices.


Here, we had to change the summary prompt to match the input as per the recommendation prompt. Although, we can use branches without changing the prompt as well. This can be easily done using `RunnableLambda`.

In [ ]:
prompt_template = ChatPromptTemplate.from_messages([("system", """You are an expert fashion stylist. Here is the complete list of available products:
                                                   {products}
                                                   Recommend the suitable products as per user request with short reasoning"""), 
                                                   ("human", "{query}")
                                                ])

# Use the original summary prompt
summary_prompt = ChatPromptTemplate.from_template("""Here are the products: {products}""")

# This lambda will extract only the products (to pass to the standard summary prompt)
extract_products = RunnableLambda(
    lambda x: {"products": x["products"]}
)

# This lambda will extract both products and query (as per the recommendation prompt template (ChatPromptTemplate, message like))
extract_full = RunnableLambda(
    lambda x: {
        "products": x["products"],
        "query": x["query"]
    }
)

In [ ]:
# Now add these lambdas to the branches
router = RunnableBranch(
    (
        lambda x: "summary" in x["query"].lower(),
        extract_products | summary_prompt | llm | StrOutputParser()
    ),
    (
        lambda x: "recommend" in x["query"].lower(),
        extract_full | prompt_template | llm | StrOutputParser()
    ),
    extract_full | llm | StrOutputParser()  # default
)

user_query = "give a short summary of products under 2000"

output = router.invoke({
    "products": df.to_string(index=False),
    "query": user_query
})

print(output)

Here’s a summary of the products you provided, organized in a clear format for easier reference:

| Product ID | Name                          | Category   | Price (INR) | Description                                      | Rating | Stock |
|------------|-------------------------------|------------|-------------|--------------------------------------------------|--------|-------|
| 101        | Classic White Sneakers        | Footwear   | 2499        | Premium white sneakers with breathable mesh.     | 4.6    | 45    |
| 102        | Slim Fit Denim Jeans          | Bottoms    | 1899        | High-quality slim fit jeans with perfect stretch.| 4.4    | 32    |
| 103        | Oversized Cotton Hoodie       | Tops       | 1499        | Ultra-soft oversized hoodie for casual style.    | 4.7    | 28    |
| 104        | Leather Crossbody Bag         | Accessories | 3299       | Elegant genuine leather crossbody bag.           | 4.8    | 15    |
| 105        | High Waist Pleated Skirt      | Bot

So,

- The `|` operator creates clean, readable pipelines
- RunnableParallel allows parallel execution of multiple components
- RunnablePassthrough and RunnableLambda give flexibility to pass data or add custom logic
- RunnableBranch executes conditional logic

### Two-Step Chain: Generate → Refine

Let's try to build a deterministic two-step pipeline for the shopping assistant:
1. Step 1: Generate initial recommendations
2. Step 2: Refine and make the response more professional and concise

In [41]:
recommend_prompt = ChatPromptTemplate.from_messages([("system", """You are an assistant. Here is the complete list of available products:
                                                   {products}
                                                   Recommend 2 suitable products as per user request with short reasoning"""), 
                                                   ("human", "{query}")
                                                ])

refine_prompt = ChatPromptTemplate.from_template("""Here is a fashion recommendation: {recommendation}
                                                 Make it more professional, concise, and customer-friendly.
                                                 Keep the same products but improve the language.""")

# Two-step chain
chain_step1 = recommend_prompt | llm | StrOutputParser()
chain_step2 = refine_prompt | llm | StrOutputParser()

# Full pipeline
full_chain = chain_step1 | chain_step2

# Execute the chain
final_response = full_chain.invoke({
    "products": df.to_string(index=False),
    "query": user_query,
    "recommendation": lambda x: x 
})

print("Final Refined Recommendation:\n")
print(final_response)

Final Refined Recommendation:

**Fashion Recommendations Under ₹2000**

Explore our selection of stylish and functional apparel, perfect for enhancing your wardrobe:

1. **Slim Fit Denim Jeans - ₹1899**  
   Discover high-quality slim fit jeans that offer exceptional stretch and comfort. Rated 4.4, with 32 pairs available.

2. **Oversized Cotton Hoodie - ₹1499**  
   Experience ultimate comfort with our ultra-soft oversized hoodie, perfect for casual outings. Rated 4.7, with 28 pieces in stock.

3. **Athletic Running Shorts - ₹999**  
   Stay active in our lightweight athletic shorts, designed for maximum performance. Rated 4.2, with 50 pairs available.

These options combine style and functionality, providing you with versatile pieces that cater to various needs. Shop now and elevate your fashion game!


## Conversations and Serialization

### Conversation History Objects

In real-world applications such as chatbots and virtual assistants, the system must remember the conversation history to provide coherent and context-aware responses.

Until now every interaction with the model was independent. The model had no memory of previous questions or answers. We used messages along with user inputs to simulate a conversation. But Langchain also provides dedicated abstractions to manage conversation state in a cleaner and more reliable way.

Conversation state (stored messages) is treated explicitly as message history, which is then wired into runnable pipelines.

In [46]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

#### `InMemoryChatMessageHistory`

Conversation state is stored using chat history implementations, the most common being `InMemoryChatMessageHistory`. This object simply stores past user and assistant messages in structured form.

In [47]:
history = InMemoryChatMessageHistory()

history.add_user_message("Hi")
history.add_ai_message("Hello! How can I help?")

history

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! How can I help?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

#### `RunnableWithMessageHistory`

We wrap our existing LCEL chain with message history management. Conversation history is injected into a runnable pipeline using `RunnableWithMessageHistory`. This wrapper ensures that past messages are automatically included at each invocation, without manual list management.

In [81]:
# Simple in-memory store for conversation sessions - we add each session's history to the dict
# In production you would use Redis, Postgres, Upstash, etc.

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [82]:
from langchain_core.prompts import MessagesPlaceholder

# Adjust prompt_template to include {history}
prompt_template = ChatPromptTemplate.from_messages([("system","""You are an assistant.Here is the complete list of available products:
                                                     {products}
                                                     Recommend 2 suitable products as per the user request with short reasoning."""),
                                                     MessagesPlaceholder(variable_name="history"),
                                                     ("human","{query}")])

conversational_chain = prompt_template | llm | StrOutputParser()

The `RunnableWithMessageHistory` object takes in several parameters

In [83]:
# Add memory layer
memory_chain = RunnableWithMessageHistory(
                            conversational_chain,
                            get_session_history,
                            input_messages_key="query",       # the key in the input dict that holds the new user message
                            history_messages_key="history"    # the key where conversation history is injected (variable we used in prompt)
                        )

In [84]:
# Use a consistent session ID to keep context across turns
session_id = "fashion_demo_session_1"

# Turn 1
response1 = memory_chain.invoke(
    {
        "products": df.to_string(index=False),
        "query": "I am a college student looking for casual jeans under 3000 rupees."
    },
    config={"configurable": {"session_id": session_id}}
)

print("Response 1:\n", response1)

Response 1:
 I recommend the following two products for you:

1. **Slim Fit Denim Jeans (Product ID: 102)** - Priced at ₹1899, these high-quality slim fit jeans offer perfect stretch, making them great for college and casual wear. With a rating of 4.4 and 32 in stock, these jeans are both stylish and comfortable.

2. **High Waist Pleated Skirt (Product ID: 105)** - Although technically a skirt, priced at ₹2199 and rated 4.3, it's a trendy and versatile option that can pair well with various casual tops. If you're open to different styles, this would be a great addition to your wardrobe.

Both options are stylish and affordable for a college student!


In [85]:
# Turn 2 – should remember previous context
response2 = memory_chain.invoke(
    {
        "products": df.to_string(index=False),
        "query": "What is the price of the first item you recommended?"
    },
    config={"configurable": {"session_id": session_id}}
)

print("Response 2:\n", response2)

Response 2:
 The price of the **Slim Fit Denim Jeans** (Product ID: 102) is ₹1899.


In [86]:
# New Session
response3 = memory_chain.invoke(
    {
        "products": df.to_string(index=False),
        "query": "What is the price of the first item you recommended?"
    },
    config={"configurable": {"session_id": "fashion_demo_session_2"}}
)

print("Response 3:\n", response3)   # Just gives the first product from the list

Response 3:
 The first item I recommended is the "Classic White Sneakers," which is priced at 2499.


In [ ]:
# The complete chat history for all sessions
store

{'fashion_demo_session_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='I am a college student looking for casual jeans under 3000 rupees.', additional_kwargs={}, response_metadata={}), AIMessage(content="I recommend the following two products for you:\n\n1. **Slim Fit Denim Jeans (Product ID: 102)** - Priced at ₹1899, these high-quality slim fit jeans offer perfect stretch, making them great for college and casual wear. With a rating of 4.4 and 32 in stock, these jeans are both stylish and comfortable.\n\n2. **High Waist Pleated Skirt (Product ID: 105)** - Although technically a skirt, priced at ₹2199 and rated 4.3, it's a trendy and versatile option that can pair well with various casual tops. If you're open to different styles, this would be a great addition to your wardrobe.\n\nBoth options are stylish and affordable for a college student!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the price of th

You can also view the chat history (for a session)

In [88]:
# Inspect current history
history = get_session_history(session_id)

print("Current conversation history for {session_id}:")

for msg in history.messages:
    print(f"{msg.type}: {msg.content[:150]}...")

Current conversation history for {session_id}:
human: I am a college student looking for casual jeans under 3000 rupees....
ai: I recommend the following two products for you:

1. **Slim Fit Denim Jeans (Product ID: 102)** - Priced at ₹1899, these high-quality slim fit jeans of...
human: What is the price of the first item you recommended?...
ai: The price of the **Slim Fit Denim Jeans** (Product ID: 102) is ₹1899....


So, `RunnableWithMessageHistory` wrapping any LCEL chain with conversation state. It uses `InMemoryChatMessageHistory` to store chat histories (across sessions)

### Serialization

Serialization in LangChain means converting prompts and runnable pipelines into a portable, inspectable representation (typically JSON). This allows you to save, version, audit, and reload LLM system structure without rewriting code.

A key design principle in modern LangChain is that structure is serializable, state is not.

Serializable components include:
* ChatPromptTemplate
* LCEL pipelines (prompt | llm | parser)
* Runnable graphs composed of prompts, models, and parsers

Serialization captures prompt text and variables, pipeline structure, and component configuration (and not conversation history and runtime outputs.)

A prompt can be serialized as follows

In [106]:
from langchain_core.load import dump

dump.default(prompt_template)

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'],
 'kwargs': {'input_variables': ['history', 'products', 'query'],
  'messages': [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['products'], input_types={}, partial_variables={}, template='You are an assistant.Here is the complete list of available products:\n                                                     {products}\n                                                     Recommend 2 suitable products as per the user request with short reasoning.'), additional_kwargs={}),
   MessagesPlaceholder(variable_name='history'),
   HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]},
 'name': 'ChatPromptTemplate'}

This produces a JSON description of message roles, templates, and placeholders (`history`, variables, etc.)


Any serializable object can be transformed in the same way. Some of the methods available are:

In [108]:
# dump.to_json_not_implemented(prompt_template)  # Serialize if object is serializable and raise error if not
# dump.default(prompt_template)                  # Serialize to json if it is a serializable object (default serialization)
# dump.dumps(prompt_template)                    # Return a JSON string
# dump.dumpd(prompt_template)                    # Return a Python dict

You can also directly use the `to_json()` method

In [109]:
serialized_prompt = prompt_template.to_json()
print(serialized_prompt)

{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'], 'kwargs': {'input_variables': ['history', 'products', 'query'], 'messages': [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['products'], input_types={}, partial_variables={}, template='You are an assistant.Here is the complete list of available products:\n                                                     {products}\n                                                     Recommend 2 suitable products as per the user request with short reasoning.'), additional_kwargs={}), MessagesPlaceholder(variable_name='history'), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]}, 'name': 'ChatPromptTemplate'}


To use the langchain object again, we can reload it, and the loaded object behaves identically to the original.

In [131]:
from langchain_core.load import load, loads

obj = load(dump.default(prompt_template),  # passing the serialized prompt
                allowed_objects=[           # for added security, you can define what objects are allowed to be loaded
                    ChatPromptTemplate,
                    AIMessage,
                    HumanMessage])
obj

ChatPromptTemplate(input_variables=['history', 'products', 'query'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated

In [ ]:
# loading a serialized chain
loaded_chain = loads(dump.dumps(conversational_chain),
                     allowed_objects='all')
loaded_chain

ChatPromptTemplate(input_variables=['history', 'products', 'query'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated

## Production-Ready Features

In this final section, we will discuss a few other tools that often help in production-level implementations:

1. LangSmith Tracing for observability
2. Streaming responses for better user experience
3. Callbacks for event captures

### LangSmith for Observability and Debugging

LangSmith is LangChain's official observability platform for tracing, debugging, and monitoring LLM applications. It helps track token usage, latency, cost, and chain execution in production.

LangSmith provides end-to-end tracing for LangChain pipelines. It records prompts, model calls, tool usage, and intermediate steps, making it easier to debug failures, analyse performance, and evaluate system behaviour over time.

In [ ]:
import os

# Enable LangSmith tracing (add these to your .env file in real projects)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "upgrad-fashion-assistant"   
os.environ["LANGCHAIN_API_KEY"] = "<your_langsmith_key>"

print("LangSmith tracing enabled. All subsequent runs will be logged.")

LangSmith tracing enabled. All subsequent runs will be logged.


LangSmith is especially useful when systems become multi-step, where failures are hard to diagnose by looking only at final outputs. By inspecting traces, you can understand what happened, in what order, and why.

### Callbacks

Callbacks expose internal execution events such as model start, token generation, and chain completion. They are used for logging, monitoring, custom metrics, and integration with external observability systems. LangSmith itself is implemented on top of the callback mechanism.

They are a hook mechanism that lets you observe and react to what happens inside an LLM pipeline while it is running. They do not change the logic of the chain; instead, they expose events such as model invocation, token generation, tool calls, and chain start/end.

#### Callback handlers

Callbacks are implemented by defining a handler class that responds to events. LangChain provides base handler interfaces that you extend.

In [146]:
from langchain_core.callbacks import BaseCallbackHandler

class SimpleLogger(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("LLM call started")

    def on_llm_end(self, response, **kwargs):
        print("LLM call finished")

In [147]:
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} briefly."
)

chain = prompt | llm | StrOutputParser()

output = chain.invoke(
    {"topic": "overfitting"},
    config={"callbacks": [SimpleLogger()]}
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LLM call started
LLM call finished


### Streaming Responses for Real-time Output

Streaming allows the assistant to show output token by token instead of waiting for the complete response. This provides a much better user experience.

Instead of `.invoke()`, we just use the `.stream()` method. It works with many runnables that `invoke()` works with (LLMs, LCEL, prompts.)

In [145]:
# Streaming example

llm = init_chat_model("gpt-4o-mini", streaming=True)

for chunk in llm.stream("Explain gradient descent"):
    print(chunk.content, end="", flush=True)

Gradient descent is an optimization algorithm commonly used in machine learning and statistics to minimize a function by iteratively moving toward the steepest descent as defined by the negative of the gradient. It is particularly used for minimizing the cost or loss function in various algorithms, especially in neural networks.

### Key Concepts:

1. **Objective Function**: In the context of machine learning, this is typically a loss function that quantifies how well a model is performing (e.g., mean squared error, cross-entropy loss).

2. **Gradient**: The gradient is a vector containing the partial derivatives of a function with respect to its parameters. It points in the direction of the steepest ascent of the function, meaning if you want to minimize the function, you should move in the opposite direction of the gradient.

3. **Learning Rate**: This is a hyperparameter that determines the size of the steps taken towards the minimum. A small learning rate may lead to slow convergen